# PubMedQA grounding comparison
Frozen protocol v1.0 · One checkpoint, three context conditions

Run sections 1–7 in order and stop at **Pause for smoke review**. Share the printed environment, checkpoint, prediction table, and validation output. Smoke results are engineering checks. Section 8 is gated by the single `RUN_MODE` setting.

The saved outputs are from the original Colab run. The default is reset to smoke mode. Section 10 audits the archived run.


## 1. Environment and run mode
Colab normally provides PyTorch, NumPy, pandas and scikit-learn. Install the tokenizer/model dependencies if needed. The GPU is selected automatically. The run manifest captures the actual environment.


In [ ]:
import importlib.util
import subprocess
import sys

missing = [p for p in ("transformers", "sentencepiece", "safetensors")
           if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


In [ ]:
import hashlib
import json
import platform
import shutil
import time
from importlib.metadata import version
from pathlib import Path
from urllib.request import urlopen
from uuid import uuid4

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

RUN_MODE = "smoke"  # Change to "full" only after reviewing the smoke output.
assert RUN_MODE in {"smoke", "full"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = False

BOOTSTRAP_SEED = 2026
PACKAGES = {name: version(name) for name in (
    "torch", "transformers", "tokenizers", "sentencepiece", "safetensors",
    "numpy", "scipy", "scikit-learn", "pandas"
)}
ENVIRONMENT = {
    "python": platform.python_version(), "platform": platform.platform(),
    "packages": PACKAGES, "device": str(DEVICE), "dtype": "float32",
    "hardware": torch.cuda.get_device_name(0) if DEVICE.type == "cuda"
                else (platform.processor() or platform.machine()),
    "cuda_build": torch.version.cuda, "bootstrap_seed": BOOTSTRAP_SEED,
}
print(json.dumps(ENVIRONMENT, indent=2))


{
  "python": "3.13.18",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "packages": {
    "torch": "2.14.1+cpu",
    "transformers": "5.22.1",
    "tokenizers": "0.23.3",
    "sentencepiece": "0.2.2",
    "safetensors": "0.8.0",
    "numpy": "2.4.6",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "pandas": "2.2.3"
  },
  "device": "cpu",
  "dtype": "float32",
  "hardware": "x86_64",
  "cuda_build": null,
  "bootstrap_seed": 2026
}


## 2. Download the official JSON files
The cell tries the official repository automatically. If access fails, upload files with the exact names below into the notebook's working directory and rerun. A local file must pass the same frozen SHA-256 check.


In [ ]:
SOURCE = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/"
HASHES = {
    "ori_pqal.json": "8b3276be8942ebbd77f3ddcda12c1749bf0e490045a736fd8438ee40cf37a41d",
    "test_ground_truth.json": "939fe566f09017d13b1ca64d2ddfee0bc2374b366048152997669cccedc44d51",
}

def sha256(data):
    return hashlib.sha256(data).hexdigest()

def dataset_file(name):
    local = Path(name)
    bundled = Path("data") / name
    if bundled.is_file() and sha256(bundled.read_bytes()) == HASHES[name]:
        return bundled.read_bytes(), "verified repository dataset"
    expected = HASHES[name]
    if local.is_file() and sha256(local.read_bytes()) == expected:
        return local.read_bytes(), "verified local file"
    try:
        with urlopen(SOURCE + name, timeout=45) as response:
            data = response.read()
        if sha256(data) != expected:
            raise ValueError("Downloaded SHA-256 differs from the frozen protocol.")
    except (OSError, ValueError) as exc:
        raise RuntimeError(f"Upload the original {name} locally and rerun this cell.") from exc
    local.write_bytes(data)
    return data, "official download"

DATA, DATA_ORIGIN = {}, {}
for name in HASHES:
    DATA[name], DATA_ORIGIN[name] = dataset_file(name)
    print(name, DATA_ORIGIN[name], sha256(DATA[name]))


ori_pqal.json official download 8b3276be8942ebbd77f3ddcda12c1749bf0e490045a736fd8438ee40cf37a41d
test_ground_truth.json official download 939fe566f09017d13b1ca64d2ddfee0bc2374b366048152997669cccedc44d51


## 3. Integrity, split and field checks
Membership uses only official test keys. The generation path uses the exact current question and a corpus of CONTEXTS bodies. The paired source abstracts are intentionally candidates in the fixed corpus. Gold values are used only by the evaluator.


In [ ]:
def unique_object(pairs):
    result = dict(pairs)
    if len(result) != len(pairs):
        raise ValueError("Duplicate JSON key")
    return result

def read_json(name):
    return json.loads(DATA[name], object_pairs_hook=unique_object)

records = read_json("ori_pqal.json")
test_keys = read_json("test_ground_truth.json").keys()
TEST_IDS = sorted(test_keys, key=int)
NON_TEST_IDS = sorted(records.keys() - test_keys, key=int)
SMOKE_IDS = NON_TEST_IDS[:6]
DEV_IDS = NON_TEST_IDS[6:26]

assert len(records) == 1000 and len(TEST_IDS) == 500
assert all(p.isascii() and p.isdecimal() and str(int(p)) == p for p in records)
assert set(TEST_IDS) <= records.keys() and len(NON_TEST_IDS) == 500
assert SMOKE_IDS == "1571683 2224269 2503176 8017535 8111516 8200238".split()
assert DEV_IDS == (
    "8245806 8262881 8422202 8985020 9003088 9044116 9107172 9140335 "
    "9191526 9278754 9347843 9363529 9381529 9444542 9446993 9550200 "
    "9569972 9602458 9645785 9745063"
).split()
assert not (set(TEST_IDS) & set(SMOKE_IDS + DEV_IDS))


In [ ]:
def context_body(parts):
    assert isinstance(parts, list) and all(isinstance(part, str) for part in parts)
    body = "\n".join(part.strip() for part in parts if part.strip())
    assert body
    return body

QUESTIONS = {pmid: row["QUESTION"] for pmid, row in records.items()}
CONTEXTS = {pmid: context_body(row["CONTEXTS"]) for pmid, row in records.items()}
assert all(isinstance(q, str) and q.strip() for q in QUESTIONS.values())
del records, test_keys
print(f"Integrity OK: {len(CONTEXTS)} documents, {len(TEST_IDS)} official test IDs")
print("Smoke IDs:", SMOKE_IDS)
print("Reserved optional prompt-check IDs:", DEV_IDS)


Integrity OK: 1000 documents, 500 official test IDs
Smoke IDs: ['1571683', '2224269', '2503176', '8017535', '8111516', '8200238']
Reserved optional prompt-check IDs: ['8245806', '8262881', '8422202', '8985020', '9003088', '9044116', '9107172', '9140335', '9191526', '9278754', '9347843', '9363529', '9381529', '9444542', '9446993', '9550200', '9569972', '9602458', '9645785', '9745063']


## 4. Load the pinned FLAN-T5-small checkpoint
The output confirms the revision, device, precision and model parameter count. The encoder input helper reserves space for the complete instruction and exact question, then clips context tokens to fit.


In [ ]:
MODEL_ID = "google/flan-t5-small"
REVISION = "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID, revision=REVISION, use_safetensors=True
).to(device=DEVICE, dtype=torch.float32).eval()

from huggingface_hub import hf_hub_download

weights = Path(hf_hub_download(
    repo_id=MODEL_ID,
    filename="model.safetensors",
    revision=REVISION,
))
assert weights.parent.name == REVISION, f"Unexpected checkpoint path: {weights}"


parameter = next(model.parameters())
assert parameter.device.type == DEVICE.type and parameter.dtype == torch.float32
assert not model.training
print(f"Loaded {MODEL_ID}@{REVISION} on {parameter.device}")
print(f"Parameters: {model.num_parameters():,}; dtype: {parameter.dtype}")


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded google/flan-t5-small@0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab on cpu
Parameters: 76,961,152; dtype: torch.float32


In [ ]:
TEMPLATE = (
    "Answer the biomedical research question with exactly one label: yes, no, or maybe.\n"
    "Question: {question}\nContext:\n"
)
INPUT_CAP = 512
GENERATION = {"do_sample": False, "num_beams": 1, "max_new_tokens": 8}
for name in ("eos_token_id", "pad_token_id", "decoder_start_token_id"):
    GENERATION[name] = getattr(model.generation_config, name)
LABELS = ["yes", "no", "maybe"]

def add_t5_special_tokens(token_ids):
    eos = tokenizer.eos_token_id
    assert eos is not None
    return token_ids + [eos]

def encode_parts(question, context):
    prefix = tokenizer.encode(
        TEMPLATE.format(question=question), add_special_tokens=False
    )
    body = tokenizer.encode(
        context, add_special_tokens=False, verbose=False
    ) if context else []

    capacity = INPUT_CAP - len(prefix) - 1  # Reserve one token for EOS.
    if capacity < 0:
        raise ValueError("The fixed instruction and question exceed the input budget.")

    kept = body[:capacity]
    input_ids = add_t5_special_tokens(prefix + kept)
    assert input_ids[:len(prefix)] == prefix and len(input_ids) <= INPUT_CAP
    return input_ids, len(body), len(kept)

def parse_label(output):
    text = output.strip().lower()
    if text.endswith((".", "!", "?")):
        text = text[:-1].rstrip()
    return text if text in LABELS else "INVALID"


In [ ]:
def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)

def predict(pmid, arm, split, run_id, context_pmid=None, top1_score=None):
    question = QUESTIONS[pmid]
    context = "" if context_pmid is None else CONTEXTS[context_pmid]
    ids, before, used = encode_parts(question, context)
    tokens = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    sync_device()
    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(input_ids=tokens, attention_mask=torch.ones_like(tokens),
                                    **GENERATION)
    sync_device()
    elapsed = (time.perf_counter() - start) * 1000
    raw = tokenizer.decode(output_ids[0].tolist(), skip_special_tokens=True,
                           clean_up_tokenization_spaces=False)
    label = parse_label(raw)
    return dict(
        run_id=run_id, protocol_version="1.0", split=split, pmid=pmid, arm=arm,
        context_pmid=context_pmid, top1_pmid=context_pmid if arm == "B" else None,
        top1_score=top1_score, zero_score=(top1_score == 0) if arm == "B" else None,
        hit_at_1=(context_pmid == pmid) if arm == "B" else None,
        question_sha256=sha256(question.encode()), context_sha256=sha256(context.encode()),
        input_ids_sha256=sha256(json.dumps(ids).encode()), input_tokens=len(ids),
        context_tokens_before=before, context_tokens_used=used, truncated=used < before,
        model_revision=REVISION, raw_output=raw, parsed_label=label,
        invalid=label == "INVALID", inference_ms=round(elapsed, 3),
    )


## 5. Closed-book smoke
Run the six fixed non-test PMIDs with an empty context slot. Expect six raw outputs and six parsed labels.


In [ ]:
SMOKE_RUN = f"smoke-{uuid4().hex[:10]}"
smoke_a = [predict(pmid, "A", "smoke", SMOKE_RUN) for pmid in SMOKE_IDS]
display(pd.DataFrame(smoke_a)[["pmid", "arm", "raw_output", "parsed_label", "input_tokens"]])


,pmid,arm,raw_output,parsed_label,input_tokens
0,1571683,A,no,no,40
1,2224269,A,no,no,35
2,2503176,A,no,no,48
3,8017535,A,no,no,44
4,8111516,A,no,no,36
5,8200238,A,no,no,40


## 6. TF–IDF and reference-context smoke
Fit one sparse index. Select the positive maximum, break exact ties by smallest numeric PMID, and return no document on zero overlap.


In [ ]:
TFIDF = dict(
    analyzer="word", lowercase=True, strip_accents=None, token_pattern=r"(?u)\b\w\w+\b",
    ngram_range=(1, 1), stop_words=None, min_df=1, max_df=1.0, max_features=None,
    binary=False, sublinear_tf=False, use_idf=True, smooth_idf=True, norm="l2",
)

def build_index(contexts):
    pmids = sorted(contexts, key=int)
    vectorizer = TfidfVectorizer(dtype=np.float64, **TFIDF)
    matrix = vectorizer.fit_transform([contexts[p] for p in pmids])
    return pmids, vectorizer, matrix

def retrieve(query, index):
    pmids, vectorizer, matrix = index
    query_vector = vectorizer.transform([query])
    scores = (query_vector @ matrix.T).tocsr()
    best = float(scores.data.max()) if scores.nnz else 0.0
    if best <= 0:
        return None, 0.0
    first_column = int(scores.indices[scores.data == best].min())
    return pmids[first_column], best

INDEX = build_index(CONTEXTS)
print("Index shape and dtype:", INDEX[2].shape, INDEX[2].dtype)


Index shape and dtype: (1000, 13583) float64


In [ ]:
def predict_retrieved(pmid, split, run_id):
    selected, score = retrieve(QUESTIONS[pmid], INDEX)
    return predict(pmid, "B", split, run_id, selected, score)

smoke_b = [predict_retrieved(pmid, "smoke", SMOKE_RUN) for pmid in SMOKE_IDS]
smoke_c = [predict(pmid, "C", "smoke", SMOKE_RUN, pmid) for pmid in SMOKE_IDS]
smoke = pd.DataFrame(smoke_a + smoke_b + smoke_c)
display(smoke[["pmid", "arm", "context_pmid", "top1_score", "hit_at_1",
               "raw_output", "parsed_label", "input_tokens", "truncated", "inference_ms"]])


,pmid,arm,context_pmid,top1_score,hit_at_1,raw_output,parsed_label,input_tokens,truncated,inference_ms
0,1571683,A,None,NaN,None,no,no,40,False,1578.862
1,2224269,A,None,NaN,None,no,no,35,False,93.452
2,2503176,A,None,NaN,None,no,no,48,False,139.896
3,8017535,A,None,NaN,None,no,no,44,False,159.569
4,8111516,A,None,NaN,None,no,no,36,False,128.876
5,8200238,A,None,NaN,None,no,no,40,False,134.592
6,1571683,B,1571683,0.363621,True,no,no,226,False,273.130
7,2224269,B,2224269,0.550706,True,yes,yes,155,False,191.278
8,2503176,B,2503176,0.442611,True,yes,yes,512,True,768.603
9,8017535,B,8017535,0.397430,True,yes,yes,254,False,423.542


## 7. Validate the pipeline using smoke outputs
The checks cover rank order and ties, forced truncation, label parsing, the PMID join, arm isolation, and repeatability. Forced fixtures exercise branches even when the six real questions do not.


In [ ]:
def score_predictions(predictions, gold):
    assert set(predictions["pmid"]) == set(gold), "Prediction and gold PMID sets differ."
    assert set(gold.values()) <= set(LABELS), "Unexpected gold label."
    assert not predictions.duplicated(["pmid", "arm"]).any()
    scored = predictions.copy()
    scored["gold_label"] = scored["pmid"].map(gold)
    assert scored["gold_label"].notna().all()
    scored["correct"] = scored["parsed_label"].eq(scored["gold_label"])
    return scored

def label_metrics(scored):
    return {
        "n": len(scored),
        "accuracy": float(scored["correct"].mean()),
        "macro_f1": float(f1_score(scored["gold_label"], scored["parsed_label"],
                                   labels=LABELS, average="macro", zero_division=0)),
        "invalid_n": int(scored["invalid"].sum()),
        "invalid_rate": float(scored["invalid"].mean()),
    }


In [ ]:
def check_fixtures():
    toy = build_index({"10": "apple", "2": "apple", "30": "banana"})
    assert retrieve("apple", toy) == ("2", 1.0)
    assert retrieve("banana", toy) == ("30", 1.0)
    assert retrieve("pear", toy) == (None, 0.0)
    q = QUESTIONS[SMOKE_IDS[0]]
    extra = " evidence" * 1000
    ids, before, kept = encode_parts(q, extra)
    prefix = tokenizer.encode(TEMPLATE.format(question=q), add_special_tokens=False)
    evidence = tokenizer.encode(extra, add_special_tokens=False, verbose=False)
    assert before > kept > 0 and len(ids) == 512
    assert ids == add_t5_special_tokens(prefix + evidence[:kept])
    assert encode_parts(q, "")[0] == add_t5_special_tokens(prefix)
    assert [parse_label(x) for x in (" YES! ", "no", "maybe?", "", "yes because", "yes..")] == [
        "yes", "no", "maybe", "INVALID", "INVALID", "INVALID"
    ]
    fixture = pd.DataFrame({
        "pmid": ["30", "10", "20"], "arm": ["A"] * 3,
        "parsed_label": ["yes", "yes", "INVALID"], "invalid": [False, False, True],
    })
    scored = score_predictions(fixture, {"20": "maybe", "10": "no", "30": "yes"})
    assert scored["correct"].tolist() == [True, False, False]
    metrics = label_metrics(scored)
    assert metrics["accuracy"] == 1 / 3 and metrics["invalid_n"] == 1
    assert np.isclose(metrics["macro_f1"], 2 / 9)
    print("PASS: TF–IDF match/tie/zero, forced truncation, parser, shuffled PMID scoring.")

check_fixtures()


PASS: TF–IDF match/tie/zero, forced truncation, parser, shuffled PMID scoring.


In [ ]:
def validate_arms(predictions, pmids):
    assert set(zip(predictions["pmid"], predictions["arm"])) == {
        (pmid, arm) for pmid in pmids for arm in "ABC"
    }
    assert len(predictions) == 3 * len(pmids)
    assert predictions["input_tokens"].le(INPUT_CAP).all()
    counts = {"source_hits": 0, "zero_queries": 0}
    for pmid, rows in predictions.groupby("pmid"):
        a, b, c = (rows.set_index("arm").loc[arm] for arm in "ABC")
        selected = b.top1_pmid if pd.notna(b.top1_pmid) else None
        assert a.context_pmid is None and c.context_pmid == pmid
        assert selected == b.context_pmid
        assert (selected, b.top1_score) == retrieve(QUESTIONS[pmid], INDEX)
        assert a.question_sha256 == b.question_sha256 == c.question_sha256
        for row in (a, b, c):
            context = "" if row.context_pmid is None else CONTEXTS[row.context_pmid]
            assert row.context_sha256 == sha256(context.encode())
            assert row.invalid == (row.parsed_label == "INVALID")
        assert bool(b.hit_at_1) == (selected == pmid)
        assert bool(b.zero_score) == (selected is None)
        if b.hit_at_1:
            counts["source_hits"] += 1
            assert (b.input_ids_sha256, b.parsed_label) == (c.input_ids_sha256, c.parsed_label)
        if b.zero_score:
            counts["zero_queries"] += 1
            assert (b.input_ids_sha256, b.parsed_label) == (a.input_ids_sha256, a.parsed_label)
    return counts

smoke_counts = validate_arms(smoke, SMOKE_IDS)
repeat = predict(SMOKE_IDS[0], "A", "smoke", SMOKE_RUN)
assert repeat["raw_output"] == smoke_a[0]["raw_output"]
# Force a source hit and an empty-retrieval fixture through the same inference path.
fixture_pmid = SMOKE_IDS[0]
fixture_hit = predict(fixture_pmid, "B", "smoke", SMOKE_RUN, fixture_pmid, 1.0)
fixture_zero = predict(fixture_pmid, "B", "smoke", SMOKE_RUN, top1_score=0.0)
for actual, expected in [(fixture_hit, smoke_c[0]), (fixture_zero, smoke_a[0])]:
    assert all(actual[k] == expected[k] for k in ("input_ids_sha256", "parsed_label"))
print("PASS: 18 paired rows, repeat inference, and forced hit/zero input-output equality.")
print("Observed smoke retrieval branches:", smoke_counts)


PASS: 18 paired rows, repeat inference, and forced hit/zero input-output equality.
Observed smoke retrieval branches: {'source_hits': 5, 'zero_queries': 0}


In [ ]:
def evaluation_gold(split):
    if split == "test":
        return read_json("test_ground_truth.json")
    ids = SMOKE_IDS if split == "smoke" else DEV_IDS
    data = read_json("ori_pqal.json")
    return {pmid: data[pmid]["final_decision"] for pmid in ids}

def manifest(run_id, split, ids):
    return {
        "run_id": run_id, "protocol_version": "1.0", "split": split, "run_ids_in_order": ids,
        "split_ids": {"smoke": SMOKE_IDS, "optional_dev": DEV_IDS, "official_test": TEST_IDS},
        "model_id": MODEL_ID, "model_revision": REVISION, "dataset_sha256": HASHES,
        "dataset_source": SOURCE, "dataset_origin": DATA_ORIGIN, "environment": ENVIRONMENT,
        "prompt_prefix": TEMPLATE, "tfidf": TFIDF, "tfidf_dtype": "float64",
        "truncation": "preserve full prefix; keep leading context tokens; add specials once",
        "max_input_tokens": INPUT_CAP, "batch_size": 1, "generation": GENERATION,
    }

def save_predictions(rows, run_id, split, ids):
    out = Path(run_id)
    out.mkdir(exist_ok=True)
    rows.to_csv(out / "predictions.csv", index=False)
    (out / "manifest.json").write_text(json.dumps(manifest(run_id, split, ids), indent=2))
    return out

smoke_dir = save_predictions(smoke, SMOKE_RUN, "smoke", SMOKE_IDS)
smoke_scored = score_predictions(smoke, evaluation_gold("smoke"))
smoke_scored.to_csv(smoke_dir / "scored_predictions.csv", index=False)
display(smoke_scored.pivot(index="pmid", columns="arm", values="parsed_label")
        .join(smoke_scored.groupby("pmid")["gold_label"].first()))
display(pd.DataFrame({arm: label_metrics(part) for arm, part in smoke_scored.groupby("arm")}).T)
print("PASS: smoke labels joined by PMID. Smoke artifacts:", smoke_dir)


,A,B,C,gold_label
pmid,,,,
1571683,no,no,no,maybe
2224269,no,yes,yes,yes
2503176,no,yes,yes,yes
8017535,no,yes,yes,yes
8111516,no,yes,yes,yes
8200238,no,no,no,no


,n,accuracy,macro_f1,invalid_n,invalid_rate
A,6.0,0.166667,0.095238,0.0,0.0
B,6.0,0.833333,0.555556,0.0,0.0
C,6.0,0.833333,0.555556,0.0,0.0


PASS: smoke labels joined by PMID. Smoke artifacts: smoke-60f3630eed


## Pause for smoke review
Stop here and share the checkpoint line, smoke tables, and PASS lines or errors. The six smoke IDs are outside the official test set. Keep `RUN_MODE = "smoke"` while reviewing. Optional prompt checks may use only the printed `DEV_IDS` if needed. Record any amendment before starting the final run.


## 8. Full run
After smoke review, change `RUN_MODE` to `"full"` and run in order. This cell generates 1,500 rows. It saves complete predictions and the manifest before section 9 joins official label values. If execution fails, fix the runtime issue and rerun the cell with the frozen settings.


In [ ]:
full_rows = full_dir = None
if RUN_MODE == "full":
    test_run = f"test-{uuid4().hex[:10]}"
    rows = []
    for number, pmid in enumerate(TEST_IDS, start=1):
        rows.append(predict(pmid, "A", "test", test_run))
        rows.append(predict_retrieved(pmid, "test", test_run))
        rows.append(predict(pmid, "C", "test", test_run, pmid))
        if number % 50 == 0:
            print(f"Generated {number}/500 question triples", flush=True)
    full_rows = pd.DataFrame(rows)
    print("Final input and pairing checks:", validate_arms(full_rows, TEST_IDS))
    full_dir = save_predictions(full_rows, test_run, "test", TEST_IDS)
    print("Saved 1,500 predictions:", full_dir)
else:
    print("Smoke mode. Official test inference skipped.")


Generated 50/500 question triples
Generated 100/500 question triples
Generated 150/500 question triples
Generated 200/500 question triples
Generated 250/500 question triples
Generated 300/500 question triples
Generated 350/500 question triples
Generated 400/500 question triples
Generated 450/500 question triples
Generated 500/500 question triples
Final input and pairing checks: {'source_hits': 433, 'zero_queries': 0}
Saved 1,500 predictions: test-b9a5965fbf


## 9. Final metrics and exports
Join labels only after the predictions are saved. Report arm metrics, confusion matrices, paired changes, the primary bootstrap interval and source-hit diagnostics. The bundle includes the manifest, predictions and all final tables.


In [ ]:
def paired_change(wide, first, second):
    diff = wide[first].astype(int) - wide[second].astype(int)
    return {"contrast": f"{first}-{second}", "n": len(diff),
            "wrong_to_right": int((diff == 1).sum()), "right_to_wrong": int((diff == -1).sum()),
            "change_pp": float(100 * diff.mean()) if len(diff) else None}

def primary_ci(wide):
    diff = wide["B"].astype(int).to_numpy() - wide["A"].astype(int).to_numpy()
    sampled = np.random.default_rng(BOOTSTRAP_SEED).integers(0, len(diff), (10_000, len(diff)))
    return np.percentile(100 * diff[sampled].mean(axis=1), [2.5, 97.5]).tolist()

def stratum_summary(wide, group):
    summary = {"group": group, "n": len(wide)}
    for arm in "ABC":
        summary[f"accuracy_{arm}"] = float(wide[arm].mean()) if len(wide) else None
    ba, cb = paired_change(wide, "B", "A"), paired_change(wide, "C", "B")
    summary.update(ba_pp=ba["change_pp"], cb_pp=cb["change_pp"],
                   cb_wrong_to_right=cb["wrong_to_right"], cb_right_to_wrong=cb["right_to_wrong"])
    b_wrong = int((~wide["B"]).sum())
    summary.update(b_wrong=b_wrong, reference_recovery=(
        cb["wrong_to_right"] / b_wrong if group == "miss" and b_wrong else None
    ))
    return summary


In [ ]:
def final_tables(scored):
    wide = scored.pivot(index="pmid", columns="arm", values="correct")
    wide = wide.reindex(TEST_IDS)
    retrieval = scored.loc[scored["arm"] == "B"].set_index("pmid").loc[TEST_IDS]
    hit = retrieval["hit_at_1"].astype(bool)
    zero = retrieval["zero_score"].astype(bool)
    arms = pd.DataFrame([{"arm": arm, **label_metrics(part)}
                         for arm, part in scored.groupby("arm")])
    contrasts = pd.DataFrame([paired_change(wide, first, second)
                              for first, second in [("B", "A"), ("C", "A"), ("C", "B")]])
    lo, hi = primary_ci(wide)
    contrasts["ci_2_5_pp"] = [lo, None, None]
    contrasts["ci_97_5_pp"] = [hi, None, None]
    strata = pd.DataFrame([stratum_summary(group, name) for name, group in (
        ("hit", wide.loc[hit]), ("miss", wide.loc[~hit]), ("all_zero", wide.loc[zero])
    )])
    diagnostics = pd.DataFrame([{"n": len(hit), "hits": int(hit.sum()), "hit_at_1": hit.mean(),
                                 "zero_n": int(zero.sum()), "zero_rate": zero.mean()}])
    return {"arm_metrics": arms, "paired_changes": contrasts,
            "hit_miss": strata, "retrieval_diagnostics": diagnostics}


In [ ]:
if RUN_MODE == "full":
    assert full_dir is not None and (full_dir / "predictions.csv").is_file()
    assert len(full_rows) == 1500
    scored = score_predictions(full_rows, evaluation_gold("test"))
    scored.to_csv(full_dir / "scored_predictions.csv", index=False)
    for table_name, table in final_tables(scored).items():
        display(table)
        table.to_csv(full_dir / f"{table_name}.csv", index=False)
    for arm, part in scored.groupby("arm"):
        confusion = pd.crosstab(part["gold_label"], part["parsed_label"]).reindex(
            index=LABELS, columns=LABELS + ["INVALID"], fill_value=0
        )
        confusion.to_csv(full_dir / f"confusion_{arm}.csv")
    bundle = shutil.make_archive(str(full_dir), "zip", root_dir=full_dir)
    from IPython.display import FileLink
    display(FileLink(bundle))
else:
    print("Smoke mode. Official test scoring and exports skipped.")


,arm,n,accuracy,macro_f1,invalid_n,invalid_rate
0,A,500,0.346,0.178947,0,0.0
1,B,500,0.528,0.342743,0,0.0
2,C,500,0.534,0.336900,0,0.0


,contrast,n,wrong_to_right,right_to_wrong,change_pp,ci_2_5_pp,ci_97_5_pp
0,B-A,500,200,109,18.2,11.4,24.8
1,C-A,500,212,118,18.8,NaN,NaN
2,C-B,500,16,13,0.6,NaN,NaN


,group,n,accuracy_A,accuracy_B,accuracy_C,ba_pp,cb_pp,cb_wrong_to_right,cb_right_to_wrong,b_wrong,reference_recovery
0,hit,433,0.348730,0.533487,0.533487,18.475751,0.000000,0,0,202,NaN
1,miss,67,0.328358,0.492537,0.537313,16.417910,4.477612,16,13,34,0.470588
2,all_zero,0,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN


,n,hits,hit_at_1,zero_n,zero_rate
0,500,433,0.866,0,0.0


/content/test-b9a5965fbf.zip

## 10. Regenerate metrics and paper figure from the archived run

For an audit without inference, run the definition cells in Section 7 beginning with `def score_predictions` and in Section 9 beginning with `def paired_change` and `def final_tables`. Then run both cells below in the repository directory. The archive's labels and manifest are used for this offline check.

In [ ]:
import json
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

LABELS = ["yes", "no", "maybe"]
BOOTSTRAP_SEED = 2026
ARCHIVE = Path("results/test-b9a5965fbf.zip")
with ZipFile(ARCHIVE) as archive:
    recorded_manifest = json.load(archive.open("manifest.json"))
    saved = pd.read_csv(archive.open("scored_predictions.csv"), dtype={"pmid": str})
    unscored = pd.read_csv(archive.open("predictions.csv"), dtype={"pmid": str})

TEST_IDS = recorded_manifest["split_ids"]["official_test"]
assert len(TEST_IDS) == len(set(TEST_IDS)) == 500
assert len(saved) == 1500 and saved["split"].eq("test").all()
assert list(zip(saved.pmid, saved.arm)) == [(pmid, arm) for pmid in TEST_IDS for arm in "ABC"]
pd.testing.assert_frame_equal(saved[unscored.columns], unscored)
assert saved.groupby("pmid").gold_label.nunique().eq(1).all()
assert saved.model_revision.eq(recorded_manifest["model_revision"]).all()
saved_gold = saved.groupby("pmid").gold_label.first().to_dict()
rescored = score_predictions(unscored, saved_gold)
assert rescored.correct.equals(saved.correct)

output = Path("regenerated")
output.mkdir(exist_ok=True)
tables = final_tables(rescored)
with ZipFile(ARCHIVE) as archive:
    for name, table in tables.items():
        original = pd.read_csv(archive.open(f"{name}.csv"))
        pd.testing.assert_frame_equal(table, original, check_dtype=False)
        table.to_csv(output / f"{name}.csv", index=False)
        print(name)
        print(table.to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt

labels = ["Closed-book", "TF–IDF top-1", "Reference context"]
colors = ["#61738B", "#127D82", "#366092"]
fig, (ax, bx) = plt.subplots(1, 2, figsize=(10.8, 4.5))
fig.subplots_adjust(left=.19, right=.97, top=.78, bottom=.24, wspace=.62)
accuracy = tables["arm_metrics"].set_index("arm").loc[list("ABC"), "accuracy"] * 100
bars = ax.barh(labels, accuracy, color=colors, height=.5)
ax.bar_label(bars, labels=[f"{v:.1f}%" for v in accuracy], padding=4)
ax.invert_yaxis()
ax.set_title("A  Accuracy on 500 questions", loc="left", fontweight="bold")

groups = tables["hit_miss"].set_index("group").loc[["hit", "miss"]]
for arm, offset, color, label in zip("BC", [-.14, .14], colors[1:], labels[1:]):
    values = groups[f"accuracy_{arm}"].to_numpy() * 100
    bars = bx.barh(np.arange(2) + offset, values, height=.23, color=color, label=label)
    bx.bar_label(bars, labels=[f"{v:.1f}%" for v in values], padding=4, fontsize=9)
bx.set_yticks([0, 1], [f"Hit ({groups.n.iloc[0]})", f"Miss ({groups.n.iloc[1]})"])
bx.invert_yaxis()
bx.set_title("B  Retrieved vs. reference", loc="left", fontweight="bold")
bx.legend(loc="lower center", bbox_to_anchor=(.45, -.31), frameon=False, ncol=2, fontsize=8)
for axis in (ax, bx):
    axis.set(xlim=(0, 70), xlabel="Accuracy (%)")
    for spine in ("top", "right"):
        axis.spines[spine].set_visible(False)
primary = tables["paired_changes"].iloc[0]
fig.suptitle("PubMedQA evidence comparison", x=.19, y=.95, ha="left", fontweight="bold")
fig.text(.19, .86, f"TF–IDF vs. closed-book: {primary.change_pp:+.1f} points "
         f"(95% paired bootstrap interval {primary.ci_2_5_pp:.1f} to {primary.ci_97_5_pp:.1f})")
fig.savefig(output / "pubmedqa-results.pdf")
fig.savefig(output / "pubmedqa-results.png", dpi=240)
plt.show()
